# 小望遠鏡の実測データから 50cm 望遠鏡での写りを予測する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hsameshima/SmallTelescope_test01/blob/main/2026/notebooks/m57_practicum.ipynb)

2026年10月8日にミューロン210（口径 210 mm）＋ ZWO ASI183MM で撮った M57（環状星雲）のデータから、同じカメラを国立天文台の 50cm 公開望遠鏡に付けたときのピクセルスケール・信号・S/N を予測し、本番の観測スケジュールを作ります。

**最初にやること**：メニューの「ファイル → ドライブにコピーを保存」で自分用のコピーを作ってください。コピーしないと作業内容が保存されません。

**進め方**：上のセルから順に実行します。`# TODO` と書かれたセルは自分でコードを書き、「答え」のセルには文章で答えます。提出は「ファイル → ダウンロード → .ipynb をダウンロード」で行います。

## 0. 準備（実行するだけ）

データをダウンロードし、解析に使う道具（関数）を読み込みます。セッションが切れたら、ここからやり直してください。

In [ ]:
import os
REPO = "SmallTelescope_test01"
YEAR = "2026"                                   # 実習の年度（リポジトリのフォルダ名）
if os.path.exists(REPO):
    !git -C {REPO} pull -q                      # 前に取得したものがあれば、最新の版にする
else:
    !git clone -q --depth 1 --branch main https://github.com/hsameshima/SmallTelescope_test01.git
DATA = os.path.join(REPO, YEAR, "data", "20261008")
if not os.path.isdir(DATA):
    raise FileNotFoundError(f"{DATA} が見つかりません。メニューの「ランタイム → ランタイムを接続解除して削除」のあと、もう一度実行してください")
print(len(os.listdir(DATA)), "files in", DATA)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from scipy import ndimage as ndi

plt.rcParams["figure.dpi"] = 100

def files(filt):
    """フィルター名（'Ha', 'O3', 'S2', 'R', 'G', 'B', 'Dark'）のファイル一覧"""
    return sorted(glob.glob(os.path.join(DATA, f"*filter-{filt}_*.fit")))

def load(path):
    """FITS を読み込んで float の 2 次元配列で返す（単位 ADU）"""
    return fits.getdata(path).astype(float)

def mad_std(a):
    """中央値絶対偏差から求めた標準偏差。星やホットピクセルに強い"""
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return 1.4826 * np.median(np.abs(a - np.median(a)))

def show(img, x=None, y=None, size=None, ax=None, title=None, marks=()):
    """画像を表示する（左下が原点）。x, y, size を与えるとその周りを拡大。
    marks に [(x, y), ...] を与えると円で印を付ける"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    if size is None:
        vmin, vmax = ZScaleInterval().get_limits(img[np.isfinite(img)])
    else:
        cut = img[max(int(y - size), 0):int(y + size), max(int(x - size), 0):int(x + size)]
        vmin, vmax = np.nanpercentile(cut, [5, 99.5])
    ax.imshow(img, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    if size is not None:
        ax.set_xlim(x - size, x + size); ax.set_ylim(y - size, y + size)
    for (mx, my) in marks:
        ax.plot(mx, my, "o", mfc="none", mec="tab:orange", ms=14, mew=1.5)
    ax.set_xlabel("x [pix]"); ax.set_ylabel("y [pix]")
    if title: ax.set_title(title)
    return ax

def centroid(img, x0, y0, r=8, n_iter=5):
    """(x0, y0) 付近にある星（または天体）の重心 (x, y) を返す"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    x, y = float(x0), float(y0)
    for _ in range(n_iter):
        m = (xx - x)**2 + (yy - y)**2 <= r**2
        w = img[m] - np.median(img[(xx - x)**2 + (yy - y)**2 <= (2*r)**2])
        w = np.clip(w, 0, None)
        x, y = (xx[m] * w).sum() / w.sum(), (yy[m] * w).sum() / w.sum()
    return x, y

def aperture_mean(img, x, y, r=3):
    """(x, y) を中心とする半径 r 画素の円の中の平均値"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    return np.nanmean(img[(xx - x)**2 + (yy - y)**2 <= r**2])

def sky_level(img, xc, yc, r_in=90, r_out=140):
    """(xc, yc) の周りのリング（半径 r_in〜r_out 画素）の中央値 = 背景のレベル"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    d2 = (xx - xc)**2 + (yy - yc)**2
    return np.nanmedian(img[(d2 >= r_in**2) & (d2 <= r_out**2)])

def line_profile(img, x0, y0, angle_deg, rmax=80, width=2):
    """(x0, y0) を通り、x 軸から angle_deg 傾いた直線に沿った断面。
    r（-rmax〜+rmax 画素、0.25 画素刻み）と、幅 ±width 画素で平均した値を返す"""
    a = np.radians(angle_deg)
    r = np.arange(-rmax, rmax + 0.01, 0.25)
    prof = []
    for off in np.arange(-width, width + 0.01, 1.0):
        ys = y0 + r * np.sin(a) + off * np.cos(a)
        xs = x0 + r * np.cos(a) - off * np.sin(a)
        prof.append(ndi.map_coordinates(img, [ys, xs], order=1))
    return r, np.mean(prof, axis=0)

def _stars(img, nmax=40):
    d = img - ndi.median_filter(img, size=25)
    s = mad_std(d)
    lab, n = ndi.label(ndi.gaussian_filter(d, 1.5) > 8 * s)
    idx = np.arange(1, n + 1)
    area = ndi.sum(np.ones_like(d), lab, idx)
    flux = ndi.sum(d, lab, idx)
    com = np.array(ndi.center_of_mass(d, lab, idx))
    ok = area >= 6
    return com[ok][np.argsort(-flux[ok])[:nmax]]

def _shift(ref_stars, img):
    d = (ref_stars[:, None, :] - _stars(img)[None, :, :]).reshape(-1, 2)
    e = np.arange(-60.5, 61)
    H, _, _ = np.histogram2d(d[:, 0], d[:, 1], bins=[e, e])
    j, i = np.unravel_index(H.argmax(), H.shape)
    sel = (np.abs(d[:, 0] - (e[j] + .5)) < 1.5) & (np.abs(d[:, 1] - (e[i] + .5)) < 1.5)
    return np.median(d[sel], axis=0)

MASTER_DARK = np.mean([load(f) for f in files("Dark")], axis=0)
_REF_STARS = _stars(load(files("G")[0]) - MASTER_DARK)

def stack(filt, flat=None):
    """そのフィルターの全フレームをダーク減算し、G バンド 1 枚目に位置を合わせて平均する。
    flat にフラット（masterflat の結果）を与えると、位置を合わせる前にフラットで割る。
    単位は ADU / 画素 / 60 秒（1 枚あたり）。端の欠けた部分は NaN"""
    out = []
    for f in files(filt):
        d = load(f) - MASTER_DARK
        if flat is not None:
            d = d / flat
        dy, dx = _shift(_REF_STARS, d)
        out.append(ndi.shift(d, (dy, dx), order=3, mode="constant", cval=np.nan))
    return np.mean(out, axis=0)

FLAT_DIR = os.path.join(REPO, YEAR, "data", "20261008_flat")

def masterflat(filt):
    """10/8 の日没前の空で作ったマスターフラット（画像の中央を 1 に規格化）。filt は 'R', 'G', 'B', 'Ha', 'O3', 'S2'"""
    return fits.getdata(os.path.join(FLAT_DIR, f"masterflat_{filt}.fits"), ext=1).astype(float)

def fwhm(img, x0, y0, r=10):
    """(x0, y0) 付近の星に 2 次元ガウス関数を当てはめ、(FWHM [pix], 縦横の比, ピーク [ADU]) を返す。
    FWHM は星像が伸びた向きとそれに直交する向きの幾何平均。縦横の比が 1 より大きいほど星像が伸びている"""
    from scipy.optimize import curve_fit
    x0, y0 = centroid(img, x0, y0)
    i, j = int(round(y0)), int(round(x0))
    cut = img[i - r:i + r + 1, j - r:j + r + 1]
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    def g(c, a, x, y, sx, sy, t, b):
        X, Y = c
        u = (X - x) * np.cos(t) + (Y - y) * np.sin(t)
        v = -(X - x) * np.sin(t) + (Y - y) * np.cos(t)
        return (a * np.exp(-u**2 / (2 * sx**2) - v**2 / (2 * sy**2)) + b).ravel()
    b0 = np.median(cut)
    p, _ = curve_fit(g, (xx, yy), cut.ravel(), p0=[cut.max() - b0, x0 - j, y0 - i, 1.5, 1.5, 0.0, b0], maxfev=5000)
    sx, sy = abs(p[3]), abs(p[4])
    return 2.3548 * np.sqrt(sx * sy), max(sx, sy) / min(sx, sy), float(cut.max())

def ellipse_profile(img, x0, y0, pa_deg, q, amax=90, da=1.26):
    """(x0, y0) を中心とする楕円環で平均した動径分布。
    pa_deg：長軸の向き（x 軸からの角度。line_profile の angle_deg と同じ）、q：軸比（短半径 ÷ 長半径）、
    da：環の幅（長半径で測る。画素）。
    各環の長半径 a [pix]、環の中の平均値、環に入る画素数 を返す"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    t = np.radians(pa_deg)
    u = (xx - x0) * np.cos(t) + (yy - y0) * np.sin(t)       # 長軸方向の座標
    v = -(xx - x0) * np.sin(t) + (yy - y0) * np.cos(t)      # 短軸方向の座標
    a = np.hypot(u, v / q)                                   # その画素を通る楕円の長半径
    edges = np.arange(0, amax + da, da)
    idx = np.digitize(a.ravel(), edges) - 1
    vals = img.ravel()
    ok = np.isfinite(vals) & (idx >= 0) & (idx < len(edges) - 1)
    n = np.bincount(idx[ok], minlength=len(edges) - 1)
    tot = np.bincount(idx[ok], weights=vals[ok], minlength=len(edges) - 1)
    return 0.5 * (edges[1:] + edges[:-1]), tot / np.maximum(n, 1), n

print("準備完了")

## 予想を書き残す

スライドで見た HST の M57 の画像（青 [O III]、緑 Hα、赤 [N II]）について、どの色が内側に、どの色が外側に広がっているかを書いてください。理由は書かなくてかまいません。3日目の講義で、自分のデータを使って確かめます。

**予想：**

（ここに書く）

## 課題1 ピクセルスケールと視野

位置が分かっている星 2 個の間隔からピクセルスケールを求め、ミューロン210 の実際の焦点距離と、50cm での写り方を計算します。

| 星（Tycho-2） | 赤経 α（°） | 赤緯 δ（°） | V 等級 | 画像上のおよその位置 |
| --- | --- | --- | --- | --- |
| TYC 2642-509-1 | 283.29843 | +33.05353 | 11.1 | 右上寄り（x ≈ 886, y ≈ 807） |
| TYC 2643-1347-1 | 283.51852 | +32.88811 | 11.2 | 左下の隅付近（x ≈ 70, y ≈ 30） |

座標は Gaia DR3（元期 2016.0）の値です。

まず G バンドの 1 枚目を表示して、2 つの星を確かめます。

In [ ]:
g1 = load(files("G")[0]) - MASTER_DARK
print(g1.shape, "(縦, 横) 画素")
show(g1, title="G band, frame 1", marks=[(886, 807), (70, 30)]);

### 1-1 星の重心を測る

`centroid(画像, x の目安, y の目安)` で重心を求めてください。拡大表示（`show(g1, x, y, size=20)`）で星の中心に印が来ているかも確かめます。

In [ ]:
# TODO: 2 つの星の重心 (x1, y1), (x2, y2) を求める
x1, y1 = ...
x2, y2 = ...
print(x1, y1, x2, y2)

### 1-2 角距離とピクセルスケール

2 星の角距離 θ は次の式で求まります（α, δ はラジアンに直して使う）。

$$\cos\theta = \sin\delta_1 \sin\delta_2 + \cos\delta_1 \cos\delta_2 \cos(\alpha_1 - \alpha_2)$$

θ を秒角（″）にし、画像上の距離（画素）で割るとピクセルスケール s（″/pix）になります。

In [ ]:
ra1, dec1 = 283.29843, 33.05353
ra2, dec2 = 283.51852, 32.88811

# TODO: 角距離 theta（秒角）、画像上の距離 d_pix（画素）、ピクセルスケール s（″/pix）
theta = ...
d_pix = ...
s = ...
print(theta, d_pix, s)

### 1-3 焦点距離と F 値、50cm でのピクセルスケールと視野

Bin4 の画素サイズは p = 2.4 µm × 4 = 9.6 µm です。焦点距離は f = 206265 × p / s で求まります。

50cm 公開望遠鏡は口径 500 mm、焦点距離 6030 mm です。画像の大きさは 1368 × 918 画素です。

In [ ]:
p_mm = 9.6e-3      # 画素サイズ [mm]
nx, ny = 1368, 918

# TODO: ミューロン210 の焦点距離 f_mew [mm] と F 値 F_mew（口径 210 mm）
f_mew = ...
F_mew = ...

# TODO: 50cm のピクセルスケール s_50 [″/pix] と F 値 F_50
s_50 = ...
F_50 = ...

# TODO: 両方の視野 [′]（横 × 縦）
fov_mew = ...
fov_50 = ...

**答え：**

（ここに書く）

### 1-4 M57 の外縁の大きさ

HST の観測では、M57 の電離ガスの外縁は長半径 約 42″、短半径 約 30″ です。

1. 42″ がミューロン210 と 50cm でそれぞれ何画素になるか計算してください。
2. Hα 画像で M57 の中心を求め、長軸に沿った断面（`line_profile`）を描いて、外側で強度がピークの半分に落ちる半径を読み取ってください。断面の角度を変えて、外縁がいちばん遠くなる向きが長軸です。

In [ ]:
ha = stack("Ha")
show(ha, 745, 445, 100, title="H-alpha (+[N II]), 2 x 60 s");

In [ ]:
# TODO: M57 の中心 (xc, yc) を求める（目で見た中心を目安に centroid(ha, x, y, r=60)。r はリング全体が入る大きさにする）
xc, yc = ...

# TODO: 42″ が何画素か（ミューロン210, 50cm）
r42_mew = ...
r42_50 = ...

# TODO: 角度を変えながら断面を描き、外側の半値半径を読み取る（角度のリストを書き換えて試す）
fig, ax = plt.subplots(figsize=(9, 4))
for ang in [0, 45, 90, 135]:                         # ← x 軸からの角度 [°]
    r, prof = line_profile(ha, xc, yc, angle_deg=ang)
    ax.plot(r, prof, label=f"{ang} deg")
ax.set_xlabel("distance from centre [pix]")
ax.set_ylabel("ADU / pix / 60 s")
ax.legend()
ax.grid(alpha=0.3)

**答え：**

（ここに書く）

### 1-5 星像の大きさ（シーイング）を測る

星の像は、大気のゆらぎ（シーイング）や追尾のずれ、ピントのずれでぼけて、ある幅に広がります。この幅を、明るさがピークの半分になる幅（半値全幅、FWHM）で表します。

`fwhm(画像, x の目安, y の目安)` は、星に 2 次元のガウス関数を当てはめて、FWHM（画素）、縦横の比、ピークの値を返します。

1. G の 1 枚目で、星を 5 個以上選んで FWHM を測り、中央値をピクセルスケール s をかけて秒角（″）に直してください。ピークが 60000 ADU を超える星は飽和しているかもしれないので使いません。
2. 測った星像の幅は、50cm（1-3 のピクセルスケール）では何画素になりますか。

In [ ]:
# 星の候補（明るい順。M57 の近くと画像の端は除く）
cand = [(x, y) for y, x in _stars(g1, nmax=30)
        if np.hypot(x - xc, y - yc) > 100 and 15 < x < g1.shape[1] - 15 and 15 < y < g1.shape[0] - 15]
print(np.round(cand[:10], 1))
fw, q, pk = fwhm(g1, *cand[0])
print(f"例: FWHM = {fw:.2f} pix、縦横の比 {q:.2f}、ピーク {pk:.0f} ADU")

# TODO: 5 個以上の星で FWHM を測り、中央値を画素と秒角（× s）で出す
#       ヒント: fws = [fwhm(g1, x, y)[0] for x, y in cand[:8]] で 8 個の星の FWHM のリストができる
fwhm_arc = ...       # 星像の幅 [″]（中央値）

# TODO: その幅は 50cm（s_50）で何画素か

**答え：**（星像の幅は何″か。この幅には大気のゆらぎのほかに何が含まれていると考えられるか〔追尾のずれ、ピント〕。縦横の比も手がかりになる）

（ここに書く）

## 課題2 信号とノイズの実測

ミューロン210 での「60 秒・1 枚・1 画素あたり」の信号とノイズを実測します。

### 2-1 1 枚あたりのノイズ

同じ条件の 2 枚の差をとると、画素ごとに固定したパターン（ホットピクセルなど）が消え、ランダムなノイズだけが残ります。差の標準偏差は 1 枚のノイズの √2 倍です。ダークと各フィルターについて、1 枚のノイズ σ と、平均レベル（2 枚の平均の中央値）を求めてください。標準偏差には `mad_std` を使います。

In [ ]:
filters = ["Dark", "S2", "Ha", "O3", "R", "G", "B"]
level, sigma = {}, {}
for f in filters:
    a, b = [load(x) for x in files(f)]
    # TODO: 平均レベル level[f] と 1 枚のノイズ sigma[f]
    level[f] = ...
    sigma[f] = ...
    print(f, level[f], sigma[f])

### 2-2 ゲイン

光子（電子）の数 N はポアソン分布に従うので、分散は N に等しくなります。信号を ADU で表すと、1 ADU が何電子にあたるかをゲイン g（e⁻/ADU）とすると、平均 S（ADU）は N／g、分散は N／g² なので、σ² ＝ S／g になります。横軸に平均レベル、縦軸に σ² をとって直線を当てはめ（フォトン・トランスファー・カーブ）、傾き 1／g からゲイン g を求めてください。

In [ ]:
# 2-2 フォトン・トランスファー・カーブ：横軸に平均レベル、縦軸に σ²
L = np.array([level[f] for f in filters])          # 平均レベル [ADU]
V = np.array([sigma[f]**2 for f in filters])       # 1 枚のノイズの二乗 [ADU^2]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(L, V, "o")
for f, x, y in zip(filters, L, V):
    ax.annotate(f, (x, y), textcoords="offset points", xytext=(5, 5))   # 点にフィルター名を付ける

# TODO: np.polyfit(L, V, 1) で直線を当てはめ、傾き slope と切片 c0 を求める（polyfit は [傾き, 切片] を返す）
slope, c0 = ...
xx = np.linspace(L.min(), L.max(), 50)
ax.plot(xx, slope * xx + c0, "--")                  # 当てはめた直線
ax.set_xlabel("mean level [ADU]")
ax.set_ylabel("sigma^2 [ADU^2]")

# TODO: 傾きは 1/g なので、ゲイン g [e-/ADU] を求める
g = ...
print("g =", g, "e-/ADU")

**答え：**

（ここに書く）

### 2-3 M57 のリングの信号

各輝線の画像（2 枚を位置合わせして平均したもの）から背景を引き、リング上に半径 3 画素の開口を 4 か所以上置いて、平均の信号 S（ADU / pix / 60 s）とそのばらつきを求めてください。開口の位置は画像を拡大して自分で決めます。

**Python のメモ（辞書と内包表記）**：下のセルの `img = {f: stack(f) for f in ["Ha", "O3", "S2"]}` は、フィルター名をキー、画像を値にした「辞書」を 1 行で作る書き方（辞書内包表記）です。次の for 文と同じ意味です。

```python
img = {}                       # 空の辞書
for f in ["Ha", "O3", "S2"]:
    img[f] = stack(f)          # キー f に、その画像を入れる
```

`img["Ha"]` で Hα の画像を取り出せます。このあとの `S`、`sigma`、`snr1_mew` なども、フィルター名をキーにした辞書です。`{f: S[f] / np.sqrt(sigma[f]**2 + S[f] / g) for f in S}` のような式は、「S のすべてのキー f について、右の式を計算した値を f に対応させる」という意味です。書きにくければ、for 文で 1 つずつ入れてかまいません。

In [ ]:
img = {f: stack(f) for f in ["Ha", "O3", "S2"]}
for f in img:
    img[f] = img[f] - sky_level(img[f], xc, yc)     # 背景を引く
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, f in zip(axs, img):
    show(img[f], xc, yc, 70, ax=ax, title=f)

In [ ]:
# TODO: リング上の開口の位置を決め、各輝線の平均信号 S[f] とばらつきを求める
positions = [(..., ...), (..., ...), (..., ...), (..., ...)]
S = {}
for f in img:
    vals = [aperture_mean(img[f], x, y, r=3) for x, y in positions]
    S[f] = np.mean(vals)
    print(f, S[f], np.std(vals))

### 2-4 1 枚・1 画素あたりの S/N

$$\left(\frac{S}{N}\right)_1 = \frac{S}{\sqrt{\sigma_{\rm bg}^2 + S / g}}$$

σ_bg は 2-1 で求めたそのフィルターの 1 枚のノイズ、S／g は天体そのものの光子ノイズの分散（ADU² 単位）です。

In [ ]:
# TODO: 各輝線の 1 枚・1 画素あたりの S/N
snr1_mew = {}

**答え：**

（ここに書く）

## 課題3 50cm での信号の換算

面輝度 Σ の天体を、口径 D・焦点距離 f の望遠鏡と画素サイズ p のカメラで撮るとき、1 画素が受ける光量は「集光面積」と「1 画素が見込む空の立体角」の積に比例します。

$$N_{\rm pix} \propto \Sigma \times \frac{\pi D^2}{4} \times \left(\frac{p}{f}\right)^2 \propto \Sigma\,\frac{p^2}{F^2}, \qquad F = \frac{f}{D}$$

比はすべて「50cm ÷ ミューロン210」で求めます。面輝度 Σ と画素の大きさ p は 2 台で同じなので、比には口径 D と焦点距離 f（F 値）だけが残ります。ミューロン210 の f は 1-3 で求めた実測の値、50cm は D ＝ 500 mm、f ＝ 6030 mm を使います。

1. **1 画素あたりの信号の比**：上の式から、1 画素の信号は 1／F² に比例します。(F_mew ／ F_50)² を計算してください（`F_mew`、`F_50` は 1-3 で求めたもの）。
2. **同じ角度面積（1″ × 1″）あたりの光子数の比**：空の同じ範囲から届く光の量は、望遠鏡が光を集める面積（πD²/4）だけで決まり、焦点距離や画素にはよりません。口径の比の 2 乗を計算してください。（50cm では 1″ × 1″ に入る画素の数が増えるので、1 画素あたりの比〔1.〕とは違う値になります）
3. **星（点源）の総光量の比**：星の光は広がっていないので、集めた光はすべて星像に入ります。これも集光面積の比になります。
4. 1-5 で測った星像の幅（FWHM）は、ミューロン210 と 50cm でそれぞれ何画素ですか（1-3 のピクセルスケールを使う）。3 の総光量の比と合わせて、星の中心の画素の明るさ（ピーク）の比を見積もり、どちらの望遠鏡で星が飽和しやすいか考えてください。
5. 夜空の背景光も「広がった光源」です。50cm で 1 画素あたりの背景光はどう変わりますか。課題2で判定したノイズの内訳は変わるでしょうか。

In [ ]:
# TODO: 比を計算する
ratio_pix = ...     # 1 画素あたり
ratio_area = ...    # 同じ角度面積あたり
ratio_star = ...    # 星の総光量

# TODO: 1-5 で測った星像の幅 fwhm_arc が何画素か（ミューロン210, 50cm）と、星のピークの比
seeing_pix_mew = ...
seeing_pix_50 = ...
ratio_peak = ...

**答え：**

（ここに書く）

## 課題4 S/N と必要積分時間の予測

カメラ・露出 60 秒・センサー温度は 10/8 と同じとし、背景ノイズ σ_bg は 1 画素あたり変わらないとします。S/N は枚数の平方根に比例して伸びます。

$$n_{\rm frame} = \left(\frac{(S/N)_{\rm goal}}{(S/N)_1}\right)^2$$

1. 50cm・Bin4 での 1 枚・1 画素あたりの S/N を各輝線で予測してください。
2. 3 × 3 画素にまとめた場合（約 1″/pix）の S/N を計算してください。まとめると信号は 9 倍、分散も 9 倍になります。1-5 で測った星像の幅は、まとめた後に何画素になりますか。星像の形を保つには、幅に 2〜3 画素以上あるのが目安です。3 × 3 にまとめてよいか、4 × 4 ではどうかも考えてください。
3. 3 色合成用に、リング上で 1 画素 S/N = 20 にするのに必要な枚数を、Bin4 のままの場合と 3 × 3 にまとめた場合で求めてください。上の式の n は小数になるので、切り上げて整数の枚数にします（`int(np.ceil(n))`。`np.ceil` は切り上げ、`int` は整数に直す関数）。
4. 動径分布用に必要な枚数を見積もります（下のセル）。
    1. `ellipse_profile` で、10/8 の Hα と [S II] の画像を幅 1″ の楕円環で平均した動径分布を描きます。楕円の中心は 1-4 の (xc, yc)、長軸の向きは 1-4 で見つけた角度、軸比（短半径 ÷ 長半径）は 0.7 とします。
    2. それぞれの輝線で、動径分布のピークの明るさ S_peak と、外縁（明るさがピークの半分に落ちる位置）の長半径 a_edge を読み取ります。
    3. 50cm で、外縁の位置にある幅 1″ の楕円環に入る画素数を見積もります（楕円の周の長さ × 1″ ÷ 1 画素の面積）。
    4. その楕円環で平均して S/N = 10 にするのに必要な枚数を求めます。外縁の明るさは S_peak の半分とし、課題3の比で 50cm に換算します。
    5. 発展（任意）：楕円環で全周を平均すると、向きによる外縁の違いは見えなくなります。向きごとに、外縁に置いた動径方向 1″ × 直交方向 3″ の区画で S/N = 10 にするには何枚必要か、楕円環と比べてください。

In [ ]:
# TODO
S_50 = {}        # 50cm での信号 [ADU/pix/60s]
snr1_50 = {}     # 50cm・Bin4 の 1 枚・1 画素 S/N
snr1_50_b3 = {}  # 3x3 にまとめた場合
n_bin4 = {}      # S/N 20 に必要な枚数（Bin4）。切り上げて整数に: int(np.ceil(...))
n_b3 = {}        # S/N 20 に必要な枚数（3x3）

In [ ]:
# 4-4 楕円環で平均した動径分布（10/8 の画像。img は 2-3 で背景を引いたもの）
PA = ...        # ← 1-4 で見つけた長軸の向き [°]（x 軸からの角度）
Q = 0.7         # 軸比（短半径 ÷ 長半径）

S_peak, a_edge = {}, {}
fig, ax = plt.subplots(figsize=(9, 4))
for f in ["Ha", "S2"]:
    a_pix, mean, npix = ellipse_profile(img[f], xc, yc, PA, Q, amax=80, da=1 / s)   # 幅 1″ の楕円環
    a_arc = a_pix * s                                   # 横軸を秒角に
    ax.plot(a_arc, mean / np.nanmax(mean), label=f)     # ピークで割って、形を比べる
ax.axhline(0.5, ls="--", c="gray")                     # ピークの半分
ax.set_xlabel("elliptical radius a [arcsec]")
ax.set_ylabel("brightness / peak")
ax.legend()
ax.grid(alpha=0.3)

# TODO: 各輝線のピークの明るさ S_peak[f]（ADU/pix/60s）と、外縁の長半径 a_edge[f]（″）を読み取る
#       （ループの中で mean と a_arc から求めてもよいし、図から読んで数字を入れてもよい）

# TODO: 50cm で、外縁の位置にある幅 1″ の楕円環の画素数 n_ring
#       楕円の周の長さ ≈ 2π √((a² + b²) / 2)、b = Q a。1 画素の面積は s_50² [″²]

# TODO: その楕円環で平均した 1 枚の S/N（外縁の明るさ 0.5 × S_peak を課題3の比で 50cm に換算）と、S/N 10 に必要な枚数

**答え：**

（ここに書く）

## 課題5（宿題）M57 の観測可能時間と当日のスケジュール

11月の M57 は夕方の西空にあり、薄明が終わるとまもなく低くなります。教員が指定する本番日について、使える時間を計算し、課題4の積分時間を割り付けたスケジュールを作ります。

| 計算に使う値 | |
| --- | --- |
| 観測地（国立天文台三鷹） | 北緯 35.672°、東経 139.538°、標高 58 m |
| M57 の位置（J2000） | α = 18h53m35.1s、δ = +33°01′45″ |
| 観測を許す高度の下限 | 30°（エアマス約 2） |

**エアマス** X は、天頂を 1 としたときに、星の光が通ってくる大気の厚さです。大気を平らな層とみなすと、高度 h の星の光は大気の中を 1 / sin h 倍の長さだけ通るので、X ≈ 1 / sin h になります（h = 90° で 1、h = 30° で 2）。大気による減光や星像の悪化は X とともに大きくなるので、ここでは X ≈ 2（高度 30°）を観測の下限とします。

下のセルの `jst_min` は、16:30〜21:30 の日本時間を「0 時からの分」で 1 分ごとに並べた配列です（例：18 時 6 分 → 1086）。`sun_alt` と `m57_alt` は、その各時刻での太陽と M57 の高度［°］です。関数 `hhmm(分)` は、分を "18:06" の形の文字列にします。`hhmm(jst_min[i])` のように、配列の i 番目の時刻を表示するのに使います。

撮影の時間には、露出時間のほかに 1 枚ごとの待ち時間（読み出し・保存、フィルター交換）も含めてください。10/8 の撮影では、同じフィルターで続けて撮ると 1 枚あたり約 1 秒、フィルターを替えると約 3〜4 秒の待ち時間がありました（ファイル名に入っている撮影開始時刻の差から分かります）。導入やピント合わせにかかる時間も見込んでください。

In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz, get_sun
from astropy.utils import iers
iers.conf.auto_download = False          # 地球回転パラメータのダウンロードを省略（分単位の計算には影響しない）

OBS_DATE = "2026-11-11"                  # ← 教員が指定する本番日に書き換える
site = EarthLocation(lat=35.672 * u.deg, lon=139.538 * u.deg, height=58 * u.m)
m57 = SkyCoord("18h53m35.1s", "+33d01m45s")

jst_min = np.arange(16 * 60 + 30, 21 * 60 + 31, 1)                       # 16:30〜21:30 を 1 分刻み（日本時間）
times = Time(f"{OBS_DATE} 00:00:00") + (jst_min - 9 * 60) * u.min      # UTC に直す
frame = AltAz(obstime=times, location=site)
sun_alt = get_sun(times).transform_to(frame).alt.deg
m57_alt = m57.transform_to(frame).alt.deg

def hhmm(m):
    return f"{int(m // 60):02d}:{int(m % 60):02d}"
print("計算できました:", OBS_DATE)

### やること

下のセルで、(1)〜(4) を順に進めます。

1. **日没と薄明の終了**：太陽の高度 `sun_alt` が、日没なら −0.83°、市民・航海・天文薄明の終了ならそれぞれ −6°、−12°、−18° を初めて下回る位置 i を求めます。`sun_alt < -18` は各時刻で True / False が並んだ配列で、`np.argmax(sun_alt < -18)` は最初の True の位置を返します。時刻は `hhmm(jst_min[i])` で表示します。日没の例がセルに書いてあります。
2. **天文薄明終了時の高度とエアマス**：その時刻の M57 の高度は、配列の同じ位置 `m57_alt[i_tw]` です。エアマスは `1 / np.sin(np.radians(高度))` で計算します（`np.sin` には度ではなくラジアンを入れる）。
3. **高度 30° を下回る時刻**：`m57_alt[i_tw:]` は、天文薄明の終了から後の高度だけを取り出した配列です。`np.argmax(m57_alt[i_tw:] < 30)` は、そこから数えて何番目かを返すので、`i_tw` を足すと元の配列での位置になります。天文薄明の終了からこの時刻までが、観測に使える時間です。
4. **図で確かめる**：図のひな形はそのまま実行できます。求めた時刻が、図の線の交わるところと合っているか確かめてください。

In [ ]:
# (1) 日没と薄明の終了 ------------------------------------------------
# 例：日没（太陽の高度が -0.83° を下回る時刻）
i_set = np.argmax(sun_alt < -0.83)        # sun_alt < -0.83 が初めて True になる位置
print("日没          ", hhmm(jst_min[i_set]))

# TODO: 同じようにして、市民・航海・天文薄明の終了の位置を求め、時刻を表示する
i_civil = ...     # -6°
i_naut = ...      # -12°
i_tw = ...        # -18°（天文薄明の終了）

# (2) 天文薄明終了時の M57 の高度とエアマス ------------------------------
alt_tw = m57_alt[i_tw]                    # その時刻の M57 の高度 [°]
X_tw = ...        # TODO: エアマス 1 / sin(高度)。np.radians で度をラジアンに直してから np.sin に入れる
print(f"天文薄明の終了 {hhmm(jst_min[i_tw])}：M57 の高度 {alt_tw:.0f}°、エアマス {X_tw:.2f}")

# (3) 高度 30° を下回る時刻 ---------------------------------------------
k = ...           # TODO: np.argmax(m57_alt[i_tw:] < 30)（i_tw から数えて何番目か）
i30 = i_tw + k                            # 元の配列での位置
print(f"高度 30° を下回る {hhmm(jst_min[i30])}、観測に使える時間 {jst_min[i30] - jst_min[i_tw]} 分")

# (4) 図：M57 の高度（左の目盛り）とエアマス（右の目盛り） --------------------
airmass = 1 / np.sin(np.radians(np.clip(m57_alt, 1, 90)))   # 全時刻のエアマス（高度 1° 未満は計算しない）
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(jst_min / 60, m57_alt, label="M57 altitude")
ax.plot(jst_min / 60, sun_alt, label="Sun altitude")
ax.axhline(30, ls="--", c="gray")                    # 高度 30°（観測の下限）
ax.axvline(jst_min[i_tw] / 60, ls=":", c="gray")      # 天文薄明の終了
ax.axvline(jst_min[i30] / 60, ls=":", c="gray")       # 高度 30° を下回る時刻
ax.set_ylim(-30, 90)
ax.set_xlabel("JST [hour]")
ax.set_ylabel("altitude [deg]")
ax.legend(loc="upper right")
ax2 = ax.twinx()                                      # 右側にエアマスの目盛り
ax2.plot(jst_min / 60, airmass, c="tab:green", alpha=0.6)
ax2.set_ylim(1, 3)
ax2.set_ylabel("airmass (green)")

撮影の順番を決めるときは、次の 2 点を考えてください。

1. 高度が下がると大気減光が増え、減光は短い波長（[O III] 501 nm）ほど大きい
2. 輝線比を求めるには、フィルターごとのエアマスがそろっているほうがよい（交互に撮る方法もある）

フラットは空が明るいうちに、暗いナローバンドから撮ります。天候や機材のトラブルに備えて、予備時間も入れてください。

**答え：**

| 時刻 | 作業 | フィルター | 露出 × 枚数 | 確認すること |
| --- | --- | --- | --- | --- |
| | 機材の準備（カメラを外気になじませ、センサー温度を記録） | | | |
| （日没） | トワイライトフラット | | | |
| | 明るい星でピント合わせ | | | |
| | M57 の導入、試し撮り | | | |
| （天文薄明終了） | 本観測 | | | |
| （高度 30°） | 観測終了 | | | |
| | ダーク（同じ露出・同じ温度） | | | |